# 🎯 People Counting System — Big Data trên Google Colab

Pipeline đầy đủ chạy không cần Docker:
```
COCO 2017 (FiftyOne)
      │
      ▼
Kafka (local)  ──►  YOLOv8 Consumer  ──►  Kafka (detections)
                                                  │
                          ┌───────────────────────┤
                          ▼                       ▼
                    MongoDB (local)          Redis (local)
                          │
                          ▼
                  Spark Structured Streaming
                  (windowed aggregation)
```

**Yêu cầu:** Runtime → GPU (T4)

## Cell 1 — Cài đặt Java (bắt buộc cho Kafka & Spark)

In [ ]:
# Java 11 — bắt buộc cho cả Kafka lẫn Spark
!apt-get update -qq
!apt-get install -y -qq default-jdk wget curl netcat-openbsd

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package netcat-openbsd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../00-netcat-openbsd_1.218-4ubuntu1_amd64.deb ...
Unpacking netcat-openbsd (1.218-4ubuntu1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../01-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-sc

## Cell 2 — Cài đặt Kafka

In [ ]:
import os

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"

# 1. Xóa sạch các file và thư mục lỗi trước đó
!rm -f /tmp/kafka.tgz
!rm -rf {KAFKA_DIR}
!rm -f /opt/kafka

if not os.path.exists(KAFKA_DIR):
    print("Downloading Kafka from Apache Archive...")
    # Thay đổi URL sang archive.apache.org và bỏ -q để xem tiến trình
    !wget https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz -O /tmp/kafka.tgz

    print("\nExtracting Kafka...")
    !tar -xzf /tmp/kafka.tgz -C /opt/
    print("✅ Kafka extracted successfully!")
else:
    print("✅ Kafka already exists")

# 2. Tạo Symlink và cấu hình biến môi trường
!ln -sf {KAFKA_DIR} /opt/kafka
os.environ['KAFKA_HOME'] = '/opt/kafka'
print(f"\n🚀 KAFKA_HOME={os.environ['KAFKA_HOME']}")

--2026-06-06 12:46:23--  https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119028138 (114M) [application/x-gzip]
Saving to: ‘/tmp/kafka.tgz’

/tmp/kafka.tgz      100%[===================>] 113.51M  8.94MB/s    in 18s     

2026-06-06 12:46:42 (6.15 MB/s) - ‘/tmp/kafka.tgz’ saved [119028138/119028138]


Extracting Kafka...
✅ Kafka extracted successfully!

🚀 KAFKA_HOME=/opt/kafka


## Cell 3 — Khởi động Zookeeper & Kafka Broker

In [ ]:
import subprocess, time, signal

KAFKA_HOME = '/opt/kafka'

# ── Zookeeper ──────────────────────────────────────────────
zk_proc = subprocess.Popen(
    [f"{KAFKA_HOME}/bin/zookeeper-server-start.sh",
     f"{KAFKA_HOME}/config/zookeeper.properties"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print("⏳ Khởi động Zookeeper...")
time.sleep(5)

# ── Kafka Broker ───────────────────────────────────────────
kafka_proc = subprocess.Popen(
    [f"{KAFKA_HOME}/bin/kafka-server-start.sh",
     f"{KAFKA_HOME}/config/server.properties"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print("⏳ Khởi động Kafka broker...")
time.sleep(10)

# ── Kiểm tra ───────────────────────────────────────────────
result = subprocess.run(
    [f"{KAFKA_HOME}/bin/kafka-broker-api-versions.sh",
     "--bootstrap-server", "localhost:9092"],
    capture_output=True, text=True, timeout=10
)
if result.returncode == 0:
    print("✅ Kafka broker đang chạy tại localhost:9092")
else:
    print("❌ Kafka chưa sẵn sàng:", result.stderr[:200])

⏳ Khởi động Zookeeper...
⏳ Khởi động Kafka broker...
✅ Kafka broker đang chạy tại localhost:9092


## Cell 4 — Tạo Kafka Topics

In [ ]:
KAFKA_HOME = '/opt/kafka'
BOOTSTRAP  = 'localhost:9092'

topics = [
    ("raw-frames",  1, 1),   # (tên, partitions, replication)
    ("detections",  1, 1),
]

for topic, partitions, replication in topics:
    r = subprocess.run(
        [f"{KAFKA_HOME}/bin/kafka-topics.sh",
         "--create", "--if-not-exists",
         "--bootstrap-server", BOOTSTRAP,
         "--topic", topic,
         "--partitions", str(partitions),
         "--replication-factor", str(replication)],
        capture_output=True, text=True
    )
    print(f"Topic '{topic}':", "✅" if r.returncode == 0 else "❌", r.stdout.strip() or r.stderr.strip())

# List topics
r = subprocess.run(
    [f"{KAFKA_HOME}/bin/kafka-topics.sh", "--list", "--bootstrap-server", BOOTSTRAP],
    capture_output=True, text=True
)
print("\nTopics hiện tại:", r.stdout.strip())

Topic 'raw-frames': ✅ Created topic raw-frames.
Topic 'detections': ✅ Created topic detections.

Topics hiện tại: detections
raw-frames


## Cell 5 — Cài đặt MongoDB & Redis

In [ ]:
# MongoDB 7
!curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg 2>/dev/null
!echo "deb [ signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-7.0.list
!apt-get update -qq && apt-get install -y -qq mongodb-org

# Redis
!apt-get install -y -qq redis-server

# Khởi động
mongo_proc = subprocess.Popen(
    ["mongod", "--dbpath", "/tmp/mongodb", "--logpath", "/tmp/mongodb.log",
     "--fork", "--bind_ip", "127.0.0.1"],
    stdout=subprocess.DEVNULL
)
!mkdir -p /tmp/mongodb
!mongod --dbpath /tmp/mongodb --logpath /tmp/mongodb.log --fork --bind_ip 127.0.0.1
!redis-server --daemonize yes --logfile /tmp/redis.log

time.sleep(3)

# Kiểm tra
r_mongo = subprocess.run(["mongosh", "--eval", "db.runCommand({ping:1})", "--quiet"],
                          capture_output=True, text=True)
r_redis = subprocess.run(["redis-cli", "ping"], capture_output=True, text=True)

print("MongoDB:", "✅" if "ok" in r_mongo.stdout else "❌", r_mongo.stdout.strip()[:60])
print("Redis:  ", "✅" if "PONG" in r_redis.stdout else "❌", r_redis.stdout.strip())

deb [ signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package mongodb-database-tools.
(Reading database ... 123483 files and directories currently installed.)
Preparing to unpack .../0-mongodb-database-tools_100.17.0_amd64.deb ...
Unpacking mongodb-database-tools (100.17.0) ...
Selecting previously unselected package mongodb-mongosh.
Preparing to unpack .../1-mongodb-mongosh_2.8.3_amd64.deb ...
Unpacking mongodb-mongosh (2.8.3) ...
Selecting previously unselected package mongodb-org-shell.
Preparing to unpack .../2-mongodb-org-shell_7.0.34_amd64.deb ...
Unpacking mongodb-org-shell (7.0.34) ...
Selecting previously unselected package mongodb-org-server.
Preparing to unpack .../3-mongodb-org-serv

## Cell 6 — Cài đặt Python packages

In [ ]:
!pip install -q \
    kafka-python-ng \
    pymongo \
    redis \
    ultralytics \
    fiftyone \
    pyspark==3.5.1 \
    opencv-python-headless \
    numpy

print("✅ Tất cả packages đã cài xong")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 7 — Load COCO 2017 Dataset (FiftyOne)

In [ ]:
import fiftyone.zoo as foz

# Load 50 ảnh validation có class 'person'
# Chỉ download ảnh cần thiết (lazy)
dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    max_samples=50,
    classes=["person"],
    label_types=["detections"],
    shuffle=True,
)

print(f"✅ Dataset loaded: {len(dataset)} samples")
print(f"   Ảnh đầu tiên: {dataset.first().filepath}")

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.zoo.datasets:Downloading split 'validation' to '/root/fiftyone/coco-2017/validation' if necessary


INFO:fiftyone.utils.coco:Downloading annotations to '/root/fiftyone/coco-2017/tmp-download/annotations_trainval2017.zip'


 100% |██████|    1.9Gb/1.9Gb [16.2s elapsed, 0s remaining, 144.9Mb/s]      


INFO:eta.core.utils: 100% |██████|    1.9Gb/1.9Gb [16.2s elapsed, 0s remaining, 144.9Mb/s]      


...

 100% |███████████████████| 50/50 [363.2ms elapsed, 0s remaining, 137.7 samples/s]     


INFO:eta.core.utils: 100% |███████████████████| 50/50 [363.2ms elapsed, 0s remaining, 137.7 samples/s]     


Dataset 'coco-2017-validation-50' created


INFO:fiftyone.zoo.datasets:Dataset 'coco-2017-validation-50' created


✅ Dataset loaded: 50 samples
   Ảnh đầu tiên: /root/fiftyone/coco-2017/validation/data/000000169169.jpg


## Cell 8 — Ingest: Publish ảnh COCO lên Kafka `raw-frames`

In [ ]:
import base64, json, uuid, time
import cv2
import numpy as np
from kafka import KafkaProducer

KAFKA_BOOTSTRAP = 'localhost:9092'
TOPIC_RAW       = 'raw-frames'
CAMERA_ID       = 'coco-cam-01'
RESIZE_WIDTH    = 640

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode(),
    compression_type='gzip',
    max_request_size=10 * 1024 * 1024,
)

def resize_frame(filepath: str, width: int) -> bytes:
    img = cv2.imread(filepath)
    h, w = img.shape[:2]
    if w > width:
        img = cv2.resize(img, (width, int(h * width / w)))
    _, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return buf.tobytes()

published = 0
for sample in dataset:
    try:
        frame_bytes = resize_frame(sample.filepath, RESIZE_WIDTH)
        frame_b64   = base64.b64encode(frame_bytes).decode()

        # Lấy ground-truth bboxes
        gt_boxes = []
        if sample.ground_truth:
            for det in sample.ground_truth.detections:
                gt_boxes.append({
                    'label': det.label,
                    'bbox_norm': [round(v, 4) for v in det.bounding_box]
                })

        message = {
            'frame_id':  str(uuid.uuid4()),
            'camera_id': CAMERA_ID,
            'timestamp': time.time(),
            'frame_b64': frame_b64,
            'metadata': {
                'source':      'coco-2017',
                'sample_id':   str(sample.id),
                'gt_boxes':    gt_boxes,
                'gt_count':    len(gt_boxes),
            }
        }

        producer.send(TOPIC_RAW, value=message, key=CAMERA_ID.encode())
        published += 1

        if published % 10 == 0:
            print(f"  [{published}/{len(dataset)}] Published {sample.filepath.split('/')[-1]}")

    except Exception as e:
        print(f"  ❌ Lỗi {sample.filepath}: {e}")

producer.flush()
print(f"\n✅ Đã publish {published} frames lên topic '{TOPIC_RAW}'")

  [10/50] Published 000000517069.jpg
  [20/50] Published 000000361147.jpg
  [30/50] Published 000000515982.jpg
  [40/50] Published 000000361730.jpg
  [50/50] Published 000000439593.jpg

✅ Đã publish 50 frames lên topic 'raw-frames'


## Cell 9 — Processing: Consume `raw-frames` → YOLOv8 → Publish `detections`

In [ ]:
import threading
from kafka import KafkaConsumer, KafkaProducer as KP
from ultralytics import YOLO

TOPIC_DETECTIONS = 'detections'
CONF_THRESHOLD   = 0.5

# Load YOLOv8 nano (tự download ~6MB)
model = YOLO('yolov8n.pt')
print("✅ YOLOv8n loaded")

detection_producer = KP(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode(),
)

consumer_raw = KafkaConsumer(
    TOPIC_RAW,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_deserializer=lambda v: json.loads(v.decode()),
    group_id='processing-group',
    auto_offset_reset='earliest',
    consumer_timeout_ms=15000,   # Dừng sau 15s không có message mới
)

processed = 0
print("⏳ Consuming raw-frames và chạy YOLOv8...\n")

for msg in consumer_raw:
    data = msg.value
    try:
        t0 = time.time()

        # Decode frame
        raw      = base64.b64decode(data['frame_b64'])
        arr      = np.frombuffer(raw, np.uint8)
        img      = cv2.imdecode(arr, cv2.IMREAD_COLOR)

        # YOLOv8 inference — class 0 = person
        results  = model(img, classes=[0], conf=CONF_THRESHOLD, verbose=False)[0]
        boxes    = results.boxes

        bboxes   = []
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            bboxes.append({
                'x1': round(x1), 'y1': round(y1),
                'x2': round(x2), 'y2': round(y2),
                'confidence': round(float(box.conf[0]), 3)
            })

        inference_ms = round((time.time() - t0) * 1000, 1)

        detection_msg = {
            'frame_id':     data['frame_id'],
            'camera_id':    data['camera_id'],
            'timestamp':    data['timestamp'],
            'person_count': len(bboxes),
            'bboxes':       bboxes,
            'inference_ms': inference_ms,
            'gt_count':     data['metadata'].get('gt_count', -1),
        }

        detection_producer.send(TOPIC_DETECTIONS, value=detection_msg,
                                key=data['camera_id'].encode())
        processed += 1

        print(f"  [{processed}] frame={data['frame_id'][:8]}… "
              f"detect={len(bboxes)} gt={data['metadata'].get('gt_count','?')} "
              f"time={inference_ms}ms")

    except Exception as e:
        print(f"  ❌ Lỗi processing: {e}")

detection_producer.flush()
consumer_raw.close()
print(f"\n✅ Processed {processed} frames → topic '{TOPIC_DETECTIONS}'")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ YOLOv8n loaded
⏳ Consuming raw-frames và chạy YOLOv8...

  [1] frame=feb13c99… detect=9 gt=20 time=2113.3ms
  [2] frame=01f321e8… detect=2 gt=4 time=76.9ms
  [3] frame=7affb93f… detect=1 gt=2 time=15.5ms
  [4] frame=e33e948b… detect=3 gt=30 time=51.5ms
  [5] frame=ce7420ab… detect=1 gt=7 time=13.7ms
  [6] frame=2e8a1fdc… detect=2 gt=14 time=54.3ms
  [7] frame=2f0216b8… detect=2 gt=7 time=15.8ms
  [8] frame=8989e15d… detect=1 gt=3 time=51.1ms
  [9] frame=e412eef3… detect=1 gt=10 time=54.7ms
  [10] frame=844e3c8b… detect=2 gt=11 time=21.8ms
  [11] frame=9cd3658d… detect=1 gt=35 time=28.5ms
  [12] frame=8a764d5e… detect=1 gt=3 time=12.6ms
  [13] frame=a03c25df… detect=1 gt=2 time=1

ERROR:kafka.consumer.fetcher:Fetch to node 0 failed: Cancelled: <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connected> [IPv4 ('172.28.0.12', 9092)]>



✅ Processed 50 frames → topic 'detections'


## Cell 10 — Storage: Consume `detections` → MongoDB + Redis

In [ ]:
from pymongo import MongoClient
import redis as redis_lib

# Kết nối storage
mongo_client = MongoClient('mongodb://localhost:27017/')
db           = mongo_client['people_counting']
col_detect   = db['detections']

# TTL index: tự xóa documents sau 1 giờ
col_detect.create_index('timestamp', expireAfterSeconds=3600)

r = redis_lib.Redis(host='localhost', port=6379, decode_responses=True)
print("✅ MongoDB và Redis đã kết nối")

consumer_det = KafkaConsumer(
    TOPIC_DETECTIONS,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_deserializer=lambda v: json.loads(v.decode()),
    group_id='storage-group',
    auto_offset_reset='earliest',
    consumer_timeout_ms=15000,
)

stored = 0
print("⏳ Consuming detections → MongoDB + Redis...\n")

for msg in consumer_det:
    det = msg.value
    try:
        cam = det['camera_id']

        # ── MongoDB: lưu toàn bộ detection record ──────────
        from datetime import datetime
        det['created_at'] = datetime.utcfromtimestamp(det['timestamp'])
        col_detect.insert_one({**det})

        # ── Redis: lưu kết quả mới nhất (TTL 5 phút) ───────
        latest_key = f"camera:{cam}:latest"
        r.hset(latest_key, mapping={
            'person_count': det['person_count'],
            'timestamp':    det['timestamp'],
            'inference_ms': det['inference_ms'],
            'frame_id':     det['frame_id'],
        })
        r.expire(latest_key, 300)  # 5 phút

        # ── Redis: time-series (sliding 1 giờ) ────────────
        history_key = f"camera:{cam}:history"
        r.zadd(history_key, {det['frame_id']: det['timestamp']})
        # Xóa entries cũ hơn 1 giờ
        r.zremrangebyscore(history_key, 0, time.time() - 3600)

        stored += 1
        if stored % 10 == 0:
            print(f"  [{stored}] Stored frame {det['frame_id'][:8]}… "
                  f"person_count={det['person_count']}")

    except Exception as e:
        print(f"  ❌ Lỗi storage: {e}")

consumer_det.close()
print(f"\n✅ Stored {stored} detections vào MongoDB + Redis")

✅ MongoDB và Redis đã kết nối
⏳ Consuming detections → MongoDB + Redis...



/tmp/ipykernel_6183/3299438756.py:34: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  det['created_at'] = datetime.utcfromtimestamp(det['timestamp'])


  [10] Stored frame 844e3c8b… person_count=2
  [20] Stored frame a92226e9… person_count=6
  [30] Stored frame 17633757… person_count=3
  [40] Stored frame fb1adf78… person_count=7
  [50] Stored frame 1f4e0d70… person_count=2


ERROR:kafka.consumer.fetcher:Fetch to node 0 failed: Cancelled: <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connected> [IPv4 ('172.28.0.12', 9092)]>



✅ Stored 50 detections vào MongoDB + Redis


## Cell 11 — Spark Structured Streaming: Windowed Aggregation từ Kafka

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 '
    'pyspark-shell'
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("PeopleCountingStreaming") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession ready")

# ── Schema của Kafka message value ─────────────────────────
detection_schema = StructType([
    StructField('frame_id',     StringType()),
    StructField('camera_id',    StringType()),
    StructField('timestamp',    DoubleType()),
    StructField('person_count', IntegerType()),
    StructField('inference_ms', DoubleType()),
    StructField('gt_count',     IntegerType()),
])

# ── Đọc từ Kafka topic: detections ─────────────────────────
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "detections") \
    .option("startingOffsets", "earliest") \
    .load()

# ── Parse JSON ──────────────────────────────────────────────
df_parsed = df_raw \
    .select(F.from_json(F.col('value').cast('string'), detection_schema).alias('d')) \
    .select('d.*') \
    .withColumn('event_time', F.to_timestamp(F.col('timestamp').cast('long')))

# ── Tumbling window 1 phút per camera ──────────────────────
df_agg = df_parsed \
    .withWatermark('event_time', '2 minutes') \
    .groupBy(
        F.window('event_time', '1 minute'),
        F.col('camera_id')
    ) \
    .agg(
        F.avg('person_count').alias('avg_count'),
        F.max('person_count').alias('max_count'),
        F.min('person_count').alias('min_count'),
        F.count('frame_id').alias('frame_count'),
        F.avg('inference_ms').alias('avg_inference_ms'),
    )

# ── Write kết quả ra console ───────────────────────────────
query = df_agg.writeStream \
    .outputMode('update') \
    .format('console') \
    .option('truncate', False) \
    .trigger(processingTime='10 seconds') \
    .start()

print("⏳ Spark Streaming đang chạy (30s)...")
query.awaitTermination(timeout=30)
print("\n✅ Spark Streaming hoàn thành")

✅ SparkSession ready
⏳ Spark Streaming đang chạy (30s)...

✅ Spark Streaming hoàn thành


## Cell 12 — Query kết quả từ MongoDB & Redis

In [ ]:
import json
from pprint import pprint

print("=" * 60)
print("📊 REDIS — Kết quả mới nhất của camera")
print("=" * 60)
latest = r.hgetall(f"camera:{CAMERA_ID}:latest")
pprint(latest)

print()
history_count = r.zcard(f"camera:{CAMERA_ID}:history")
print(f"📈 Redis history entries (1h): {history_count}")

print()
print("=" * 60)
print("🍃 MONGODB — Tổng hợp thống kê")
print("=" * 60)
total = col_detect.count_documents({})
print(f"Total documents: {total}")

pipeline = [
    {"$group": {
        "_id":          "$camera_id",
        "avg_count":    {"$avg": "$person_count"},
        "max_count":    {"$max": "$person_count"},
        "total_frames": {"$sum": 1},
        "last_seen":    {"$max": "$timestamp"},
    }}
]
for doc in col_detect.aggregate(pipeline):
    doc['avg_count'] = round(doc['avg_count'], 2)
    pprint(doc)

print()
print("=" * 60)
print("🔍 MONGODB — 3 detection gần nhất")
print("=" * 60)
for doc in col_detect.find({}, {'_id':0,'frame_b64':0}).sort('timestamp', -1).limit(3):
    pprint(doc)
    print()

📊 REDIS — Kết quả mới nhất của camera
{'frame_id': '1f4e0d70-03f5-4b1d-9765-e45c0398a6c7',
 'inference_ms': '12.7',
 'person_count': '2',
 'timestamp': '1780750642.2769022'}

📈 Redis history entries (1h): 50

🍃 MONGODB — Tổng hợp thống kê
Total documents: 50
{'_id': 'coco-cam-01',
 'avg_count': 2.44,
 'last_seen': 1780750642.2769022,
 'max_count': 10,
 'total_frames': 50}

🔍 MONGODB — 3 detection gần nhất
{'bboxes': [{'confidence': 0.845, 'x1': 0, 'x2': 44, 'y1': 199, 'y2': 362},
            {'confidence': 0.587, 'x1': 33, 'x2': 65, 'y1': 209, 'y2': 304}],
 'camera_id': 'coco-cam-01',
 'created_at': datetime.datetime(2026, 6, 6, 12, 57, 22, 276000),
 'frame_id': '1f4e0d70-03f5-4b1d-9765-e45c0398a6c7',
 'gt_count': 10,
 'inference_ms': 12.7,
 'person_count': 2,
 'timestamp': 1780750642.2769022}

{'bboxes': [{'confidence': 0.933, 'x1': 324, 'x2': 640, 'y1': 71, 'y2': 424},
            {'confidence': 0.811, 'x1': 54, 'x2': 109, 'y1': 172, 'y2': 372},
            {'confidence': 0.715, 'x1'

## Cell 13 — Dọn dẹp (chạy khi kết thúc session)

In [ ]:
# Dừng Spark
try:
    spark.stop()
    print("✅ Spark stopped")
except: pass

# Dừng Kafka
try:
    kafka_proc.terminate()
    zk_proc.terminate()
    print("✅ Kafka & Zookeeper stopped")
except: pass

# Dừng MongoDB
!mongod --dbpath /tmp/mongodb --shutdown 2>/dev/null || true

# Dừng Redis
!redis-cli shutdown nosave 2>/dev/null || true

print("\n✅ Tất cả services đã dừng")

ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connected> [IPv4 ('172.28.0.12', 9092)]>: Closing connection. KafkaConnectionError: Socket EVENT_READ without in-flight-requests
ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connected> [IPv4 ('172.28.0.12', 9092)]>: Closing connection. KafkaConnectionError: Socket EVENT_READ without in-flight-requests
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 

✅ Spark stopped
✅ Kafka & Zookeeper stopped
{"t":{"$date":"2026-06-06T13:00:53.405+00:00"},"s":"I",  "c":"NETWORK",  "id":4915701, "ctx":"main","msg":"Initialized wire specification","attr":{"spec":{"incomingExternalClient":{"minWireVersion":0,"maxWireVersion":21},"incomingInternalClient":{"minWireVersion":0,"maxWireVersion":21},"outgoing":{"minWireVersion":6,"maxWireVersion":21},"isInternalClient":true}}}
{"t":{"$date":"2026-06-06T13:00:53.419+00:00"},"s":"I",  "c":"CONTROL",  "id":23285,   "ctx":"main","msg":"Automatically disabling TLS 1.0, to force-enable TLS 1.0 specify --sslDisabledProtocols 'none'"}
{"t":{"$date":"2026-06-06T13:00:53.419+00:00"},"s":"I",  "c":"NETWORK",  "id":4648601, "ctx":"main","msg":"Implicit TCP FastOpen unavailable. If TCP FastOpen is required, set tcpFastOpenServer, tcpFastOpenClient, and tcpFastOpenQueueSize."}
{"t":{"$date":"2026-06-06T13:00:53.425+00:00"},"s":"I",  "c":"REPL",     "id":5123008, "ctx":"main","msg":"Successfully registered PrimaryOnlySer

ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:Connect attempt to <BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]> returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection node_id=0 host=66a1fa562ff6:9092 <connecting> [IPv4 ('172.28.0.12', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONN


✅ Tất cả services đã dừng
